In [1]:
import sys
from pathlib import Path
prediction_mode_path = Path("../module")
sys.path.append(prediction_mode_path.as_posix())
import models_creation as pred_model

import pandas as pd
import numpy as np

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from sklearn.model_selection import train_test_split
import math

import matplotlib.pyplot as plt
from sklearn.tree import plot_tree
#from sklearn import tree

from matplotlib import pyplot as plt

import joblib

from mordred import Calculator, descriptors
import mordred
from rdkit import Chem
from rdkit.Chem import PandasTools
from rdkit.Chem import Draw

import warnings
warnings.filterwarnings('ignore')

In [2]:
molecular_descriptors_df = pred_model.prepare_data('../Data/QSPR_epoxidation_model_II.xlsx')

100%|██████████████████████████████████████████████████████████████████████████████████| 67/67 [00:01<00:00, 38.75it/s]


Data size (rows, columns): (67, 3874)
Data size after second reduction (rows, columns): (67, 3874)


In [3]:
target = 'ee'
data = molecular_descriptors_df
#data[target] = load_prepared_data['ee']
data.head()

,AATS0Z,AATS0are,AATS0d,AATS0dv,AATS0i,AATS0m,AATS0p,AATS0pe,AATS0s,AATS0se,...,C_FP_2039,C_FP_2040,C_FP_2041,C_FP_2042,C_FP_2043,C_FP_2044,C_FP_2045,C_FP_2046,C_FP_2047,ee
0,21.096774,5.806774,3.096774,6.451613,154.935746,83.174801,1.660974,5.923665,3.939068,7.356725,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,92
1,32.285714,5.973175,3.321429,7.128748,153.200722,131.707465,1.861423,6.164168,4.175784,7.587078,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,55
2,32.285714,5.973175,3.321429,7.128748,153.200722,131.707465,1.861423,6.164168,4.175784,7.587078,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,97
3,22.000000,5.860000,3.142857,6.857143,153.799230,86.861521,1.707574,5.980396,4.190476,7.395751,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,97
4,21.562500,5.820625,3.375000,6.687500,154.056327,85.083842,1.696221,5.941753,3.793403,7.362468,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,74


## The best model is: Random Forest (correlation_threshold = 0.48, 2 features, 2 estimators, random_state=42) ## RF

In [4]:
model, train_r2_, test_r2_, hist1, hist2, target_column_name, training_data_RMSE, test_data_RMSE  = pred_model.prepare_data_and_create_model(molecular_descriptors_df=data, 
                                                                                                    correlation_threshold=0.48, 
                                                                                                    standardization=False, 
                                                                                                    model_type='RandomForestRegressor',
                                                                                                    target_column_name = target,
                                                                                                    n_estimators_ = 2,
                                                                                                    random_state=42,
                                                                                                    train_test_split_=True, 
                                                                                                    verbose=True)
print("R^2 score: " + str(r2_score(data[target], model.predict(data[hist2['molecular descriptor name']]))))
print("Mean squared error: "+str(mean_squared_error(data[target], model.predict(data[hist2['molecular descriptor name']]))))
print('Mean absolute error: ' + str(mean_absolute_error(data[target], model.predict(data[hist2['molecular descriptor name']]))))
print("Root Mean Square Error: "+ str(math.sqrt(mean_squared_error(data[target], model.predict(data[hist2['molecular descriptor name']])))))

I am not doing standardization...
  molecular descriptor name
0                    AATS0Z
1                  AATS0are
2                    AATS0d
3                   AATS0dv
4                    AATS0i
  molecular descriptor name  corr_value
0                    AATS0Z    0.104360
1                  AATS0are    0.076537
2                    AATS0d    0.307976
3                   AATS0dv    0.189495
4                    AATS0i   -0.304788
  molecular descriptor name  corr_value  absolute correlation value
0                    AATS0Z    0.104360                    0.104360
1                  AATS0are    0.076537                    0.076537
2                    AATS0d    0.307976                    0.307976
3                   AATS0dv    0.189495                    0.189495
4                    AATS0i   -0.304788                    0.304788
     molecular descriptor name  corr_value  absolute correlation value
489                      C2SP2    0.502393                    0.502393
3134    

In [5]:
list(hist2['molecular descriptor name'])

['C2SP2', 'C_FP_1308']

In [6]:
# save
joblib.dump(model, "models/RF_model_1__EE_reg.joblib")

['models/RF_model_1__EE_reg.joblib']

In [7]:
load_unseen_data = pd.read_excel('../Data/9_test_reactions.xlsx')
df_unseen = load_unseen_data[['SMILES', 'ee', ]]
df__ = pred_model.prepare_data('../Data/9_test_reactions.xlsx')
predicted_activity_cla = model.predict(df__[hist2['molecular descriptor name']])

100%|████████████████████████████████████████████████████████████████████████████████████| 9/9 [00:00<00:00,  9.47it/s]

Data size (rows, columns): (9, 3874)
Data size after second reduction (rows, columns): (9, 3874)


In [8]:
df_unseen['Predicted ee reg'] = model.predict(df__[hist2['molecular descriptor name']])

In [9]:
print("R^2 score: " + str(r2_score(df_unseen['ee'], df_unseen['Predicted ee reg'])))
print("Mean squared error: "+str(mean_squared_error(df_unseen['ee'], df_unseen['Predicted ee reg'])))
print('Mean absolute error: ' + str(mean_absolute_error(df_unseen['ee'], df_unseen['Predicted ee reg'])))
print("Root Mean Square Error: "+ str(math.sqrt(mean_squared_error(df_unseen['ee'], df_unseen['Predicted ee reg']))))

R^2 score: 0.4859492048915085
Mean squared error: 726.842439058957
Mean absolute error: 19.279761904761905
Root Mean Square Error: 26.960015561177947


In [10]:
df_unseen

,SMILES,ee,Predicted ee reg
0,ClC1=CC=C(C=C1)C(=O)C=CC1=CC=CC=C1,93,86.482143
1,O=C1CCCC=C1,44,17.500000
2,ClC1=C(C(C=CC2=CC=CC=C2)=O)C=CC=C1,88,86.482143
3,O=C(C=CC1=CC2=CC=CC=C2O1)C1=CC=CC=C1,99,70.750000
4,CCCCC=CC(C1=CC=CC=C1)=O,88,29.625000
5,O=C(/C1=C/C=C/C2=CC=CC=C2)C3=C(CC1)C=CC=C3,0,9.357143
6,O=C(/C=C/C1=CC=CC=C1)OCC,0,0.000000
7,O=C(C1=CC=C(N(=O)=O)C=C1)C=CC2=CC=CC=C2,48,86.482143
8,FC1=CC=C(C=CC(=O)C2=CC=C(F)C=C2)C=C1,91,86.482143


In [1]:
import sys
from pathlib import Path
prediction_mode_path = Path("../module")
sys.path.append(prediction_mode_path.as_posix())
import models_creation as pred_model

import pandas as pd
import numpy as np

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import math

import matplotlib.pyplot as plt
from sklearn.tree import plot_tree
#from sklearn import tree

from matplotlib import pyplot as plt

import joblib

from mordred import Calculator, descriptors
import mordred
from rdkit import Chem
from rdkit.Chem import PandasTools
from rdkit.Chem import Draw

import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_excel('../Data/pred_model_I_i_II.xlsx')
df.head(1)

,SMILES,0,1,2,3,4,5,6,7,8,9,9.1,9.2,9.3,9.4
0,O=C(C1=CC=CC=C1)C(O2)C2(C(F)(F)F)C3=CC=CC=C3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
molecular_descriptors_df = pred_model.prepare_data('../Data/pred_model_I_i_II.xlsx')

100%|████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00,  5.97it/s]


Data size (rows, columns): (4, 3874)
Data size after second reduction (rows, columns): (4, 3874)


In [4]:
model = joblib.load("models/RF_model_1__EE_reg.joblib")

In [5]:
predicted_activity = model.predict(molecular_descriptors_df[list(model.feature_names_in_)])

In [6]:
predicted_activity

array([ 0.        , 49.5       , 40.83333333, 49.5       ])